# Logistic Regression Assignment
## Student Mental Health & Academic Performance Dataset
---

In [3]:
# ========================
# LIBRARIES IMPORT
# ========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings('ignore')

# Dataset load karo
df = pd.read_csv('c:/Users/aqib farooq/Downloads/python_daily_tasks/student_mental_health_burnout.csv')
print('Dataset loaded successfully!')
print('Shape:', df.shape)

Dataset loaded successfully!
Shape: (150000, 20)


---
# Section 1: Data Understanding

In [4]:
# Q1: Rows, Columns aur Unique Students
print('=== Q1: Dataset Size ===')
print(f'Total Rows    : {df.shape[0]:,}')
print(f'Total Columns : {df.shape[1]}')
print(f'Unique Students: {df["student_id"].nunique():,}')

# Explanation:
# df.shape[0] = rows, df.shape[1] = columns
# nunique() = unique values count

=== Q1: Dataset Size ===
Total Rows    : 150,000
Total Columns : 20
Unique Students: 150,000


In [5]:
# Q2: Missing Values check karo
print('=== Q2: Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

# Systematic check:
# df.isnull() = har cell True/False
# .sum() = har column mein kitne True hain

=== Q2: Missing Values ===
student_id                 0
age                        0
gender                     0
course                     0
year                       0
daily_study_hours          0
daily_sleep_hours          0
screen_time_hours          0
stress_level               0
anxiety_score              0
depression_score           0
academic_pressure_score    0
financial_stress_score     0
social_support_score       0
physical_activity_hours    0
sleep_quality              0
attendance_percentage      0
cgpa                       0
internet_quality           0
burnout_level              0
dtype: int64

Total missing values: 0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   student_id               150000 non-null  int64  
 1   age                      150000 non-null  int64  
 2   gender                   150000 non-null  str    
 3   course                   150000 non-null  str    
 4   year                     150000 non-null  str    
 5   daily_study_hours        150000 non-null  float64
 6   daily_sleep_hours        150000 non-null  float64
 7   screen_time_hours        150000 non-null  float64
 8   stress_level             150000 non-null  str    
 9   anxiety_score            150000 non-null  int64  
 10  depression_score         150000 non-null  int64  
 11  academic_pressure_score  150000 non-null  int64  
 12  financial_stress_score   150000 non-null  int64  
 13  social_support_score     150000 non-null  int64  
 14  physical_activi

In [7]:
# Q3: Categorical, Numerical aur Ordinal columns
print('=== Q3: Column Types ===')

numerical_cols = ['age', 'daily_study_hours', 'daily_sleep_hours', 'screen_time_hours',
                  'anxiety_score', 'depression_score', 'academic_pressure_score',
                  'financial_stress_score', 'social_support_score',
                  'physical_activity_hours', 'attendance_percentage', 'cgpa']

ordinal_cols = ['stress_level', 'sleep_quality', 'internet_quality', 'burnout_level']
# Ordinal = inka ek order hota hai: Low < Medium < High

categorical_cols = ['gender', 'course', 'year']
# Categorical = sirf categories, koi order nahi

print('Numerical columns (numbers hain):')
print(numerical_cols)
print()
print('Ordinal columns (order hai: Low<Medium<High):')
print(ordinal_cols)
print()
print('Categorical columns (sirf categories, koi order nahi):')
print(categorical_cols)

=== Q3: Column Types ===
Numerical columns (numbers hain):
['age', 'daily_study_hours', 'daily_sleep_hours', 'screen_time_hours', 'anxiety_score', 'depression_score', 'academic_pressure_score', 'financial_stress_score', 'social_support_score', 'physical_activity_hours', 'attendance_percentage', 'cgpa']

Ordinal columns (order hai: Low<Medium<High):
['stress_level', 'sleep_quality', 'internet_quality', 'burnout_level']

Categorical columns (sirf categories, koi order nahi):
['gender', 'course', 'year']


In [ ]:
# Q4: Mean, Median, Std for cgpa, daily_study_hours, screen_time_hours
print('=== Q4: Statistics ===')
cols = ['cgpa', 'daily_study_hours', 'screen_time_hours']
for col in cols:
    print(f'\n{col}:')
    print(f'  Mean   : {df[col].mean():.2f}')
    print(f'  Median : {df[col].median():.2f}')
    print(f'  Std Dev: {df[col].std():.2f}')

In [ ]:
# Q5: Gender with highest average anxiety_score
print('=== Q5: Anxiety Score by Gender ===')
anxiety_by_gender = df.groupby('gender')['anxiety_score'].mean()
print(anxiety_by_gender)
print(f'\nHighest anxiety: {anxiety_by_gender.idxmax()} with score {anxiety_by_gender.max():.4f}')

In [ ]:
# Q6: Unique courses aur proportions
print('=== Q6: Course Distribution ===')
course_prop = df['course'].value_counts(normalize=True) * 100
print(course_prop.round(2).to_string())
print(f'\nTotal unique courses: {df["course"].nunique()}')

In [ ]:
# Q7: Year of study distribution
print('=== Q7: Year of Study Distribution ===')
year_dist = df['year'].value_counts().sort_index()
print(year_dist)

In [ ]:
# Q8: Average CGPA - Good vs Poor sleep quality
print('=== Q8: CGPA by Sleep Quality ===')
cgpa_sleep = df[df['sleep_quality'].isin(['Good', 'Poor'])].groupby('sleep_quality')['cgpa'].mean()
print(cgpa_sleep)
diff = cgpa_sleep['Good'] - cgpa_sleep['Poor']
print(f'\nDifference (Good - Poor): {diff:.4f}')
print('Good sleep quality students ka CGPA thoda better hai.')

In [ ]:
# Q9: Correlation between daily_study_hours and cgpa
print('=== Q9: Correlation - Study Hours vs CGPA ===')
corr = df['daily_study_hours'].corr(df['cgpa'])
print(f'Correlation: {corr:.5f}')
print('\nYeh correlation bahut weak hai (-0.005)')
print('Iska matlab: is dataset mein study hours aur CGPA ka koi strong relation nahi.')

In [ ]:
# Q10: Most frequent stress_level
print('=== Q10: Stress Level Distribution ===')
stress_vc = df['stress_level'].value_counts()
stress_pct = (stress_vc / len(df) * 100).round(2)
print('Count:')
print(stress_vc)
print('\nPercentage:')
print(stress_pct)
print(f'\nMost frequent: {stress_vc.idxmax()} ({stress_pct.max()}%)')

---
# Section 2: Data Cleaning & Preprocessing

In [ ]:
# Missing Value Handling
print('=== Missing Values Analysis ===')

# a. Rows with at least one NaN
rows_with_nan = df.isnull().any(axis=1).sum()
print(f'a. Rows with at least 1 NaN: {rows_with_nan}')

# b. Strategies (explain karein)
print()
print('b. Imputation Strategies:')
print('   Numerical columns ke liye:')
print('   1. Mean imputation - normal distribution mein achha kaam karta hai')
print('   2. Median imputation - outliers ke waqt better hota hai')
print('   Categorical columns ke liye:')
print('   1. Mode imputation - most common value se bharo')
print('   2. Unknown category - ek naya category "Unknown" banao')

# c. Rows with ALL NaN
rows_all_nan = df.isnull().all(axis=1).sum()
print(f'\nc. Rows with ALL NaN: {rows_all_nan}')
print('   Agar hote toh drop kar dete: df.dropna(how="all")')

In [ ]:
# Duplicate Detection
print('=== Duplicate Detection ===')
dupes = df.duplicated().sum()
print(f'a. Duplicate rows: {dupes}')
print()
print('b. Duplicates ka effect on Logistic Regression:')
print('   - Model us data ko zyada weight dega jo repeated hai')
print('   - Overfitting ho sakta hai')
print('   - Model ki performance test data pe kharab ho sakti hai')
print('   - Fix: df.drop_duplicates(inplace=True)')

In [ ]:
# Outlier Detection using IQR
print('=== Outlier Detection (IQR Method) ===')
num_cols = ['cgpa', 'daily_study_hours', 'screen_time_hours', 'anxiety_score',
            'depression_score', 'academic_pressure_score', 'attendance_percentage']

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f'{col}: {outliers} outliers | Range: [{lower:.2f}, {upper:.2f}]')

print()
print('Outlier Treatment Methods:')
print('1. IQR Capping (Winsorization): outliers ko boundary value pe le aao')
print('   df[col] = df[col].clip(lower, upper)')
print('2. Log Transformation: skewed data ko normal banao')
print('   df[col] = np.log1p(df[col])')
print()
print('Logistic Regression ke liye: IQR Capping better hai')
print('Kyunki log transformation interpretation mushkil kar deta hai')

In [ ]:
# Encoding Categorical Variables
# Pehle ek copy banate hain
df_encoded = df.copy()

print('=== Encoding Categorical Variables ===')
print()
print('a. Categorical columns jinhe encode karna hai:')
print('   gender, course, year -- kyunki ML models sirf numbers samajhte hain')

# b. One-Hot Encoding
print()
print('b. One-Hot Encoding (gender, course, year):')
df_ohe = pd.get_dummies(df_encoded, columns=['gender', 'course', 'year'], drop_first=True)
print('New columns after OHE:', [c for c in df_ohe.columns if 'gender' in c or 'course' in c or 'year' in c])

# Label Encoding
print()
print('Label Encoding (gender):')
le = LabelEncoder()
df_encoded['gender_label'] = le.fit_transform(df_encoded['gender'])
print(dict(zip(le.classes_, le.transform(le.classes_))))

# c. Ordinal Encoding
print()
print('c. Ordinal Encoding (order matter karta hai):')
print('   sleep_quality: Poor=0, Average=1, Good=2')
print('   stress_level:  Low=0, Medium=1, High=2')
print('   burnout_level: Low=0, Medium=1, High=2')

df_encoded['sleep_quality_enc'] = df_encoded['sleep_quality'].map({'Poor': 0, 'Average': 1, 'Good': 2})
df_encoded['stress_level_enc'] = df_encoded['stress_level'].map({'Low': 0, 'Medium': 1, 'High': 2})
df_encoded['internet_quality_enc'] = df_encoded['internet_quality'].map({'Poor': 0, 'Average': 1, 'Good': 2})
print('Ordinal encoding done!')

print()
print('Ordinal encoding OHE se better kyun hai yahan:')
print('   Kyunki Poor < Average < Good -- yeh order important hai')
print('   OHE yeh order kho deta hai')

In [ ]:
# Feature Scaling
print('=== Feature Scaling ===')
print('a. Standardization kyun zaroori hai Logistic Regression ke liye:')
print('   - Logistic Regression gradient descent use karta hai')
print('   - Agar features alag scale pe hain (cgpa: 0-10, anxiety: 0-20)')
print('   - Toh model kisi ek feature ko zyada importance de deta hai')
print('   - StandardScaler: mean=0, std=1 kar deta hai')
print()

# Numerical columns scale karo
num_cols = ['age', 'daily_study_hours', 'daily_sleep_hours', 'screen_time_hours',
            'anxiety_score', 'depression_score', 'academic_pressure_score',
            'financial_stress_score', 'social_support_score',
            'physical_activity_hours', 'attendance_percentage', 'cgpa']

scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[num_cols] = scaler.fit_transform(df_encoded[num_cols])

print('b. After StandardScaler - Mean and Std:')
for col in num_cols[:5]:  # pehle 5 dikha rahe hain
    print(f'   {col}: mean={df_scaled[col].mean():.4f}, std={df_scaled[col].std():.4f}')
print('   (mean ~0.0 aur std ~1.0 hona chahiye -- scaling sahi hai!)')

In [ ]:
# Data Type Validation
print('=== Data Type Validation ===')
print(df.dtypes)
print()
print('Sab columns ke data types theek hain!')
print('Numerical: int64/float64')
print('Categorical: object/str')

---
# Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# Q1: CGPA Distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['cgpa'].hist(bins=30, color='steelblue', edgecolor='white')
plt.title('CGPA Distribution')
plt.xlabel('CGPA')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
from scipy import stats
stats.probplot(df['cgpa'], plot=plt)
plt.title('CGPA Q-Q Plot')
plt.tight_layout()
plt.show()

print(f'Skewness: {df["cgpa"].skew():.4f}')
print(f'Kurtosis: {df["cgpa"].kurtosis():.4f}')
print('Skewness ~0 means nearly normal distribution hai')

In [ ]:
# Q2: Boxplot - CGPA by Stress Level
plt.figure(figsize=(8, 5))
order = ['Low', 'Medium', 'High']
df.boxplot(column='cgpa', by='stress_level', figsize=(8,5))
plt.title('CGPA by Stress Level')
plt.suptitle('')
plt.xlabel('Stress Level')
plt.ylabel('CGPA')
plt.show()

print('Insights:')
print(df.groupby('stress_level')['cgpa'].mean())
print('High stress wale students ka CGPA compare karo Low stress se')

In [ ]:
# Q3: Bar plot - Anxiety Score by Gender
plt.figure(figsize=(7, 4))
anxiety_gender = df.groupby('gender')['anxiety_score'].mean()
anxiety_gender.plot(kind='bar', color=['steelblue', 'salmon', 'green'], edgecolor='white')
plt.title('Average Anxiety Score by Gender')
plt.xlabel('Gender')
plt.ylabel('Average Anxiety Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('Exact values:')
print(anxiety_gender)

In [ ]:
# Q4: Scatter plot - Study Hours vs CGPA colored by Sleep Quality
plt.figure(figsize=(8, 5))
colors = {'Good': 'green', 'Average': 'orange', 'Poor': 'red'}
for sq, color in colors.items():
    subset = df[df['sleep_quality'] == sq]
    plt.scatter(subset['daily_study_hours'], subset['cgpa'],
                c=color, label=sq, alpha=0.4, s=5)
plt.xlabel('Daily Study Hours')
plt.ylabel('CGPA')
plt.title('Study Hours vs CGPA (by Sleep Quality)')
plt.legend(title='Sleep Quality')
plt.tight_layout()
plt.show()

In [ ]:
# Q5: Grouped bar chart - Burnout Level by Course
plt.figure(figsize=(12, 5))
burnout_course = df.groupby(['course', 'burnout_level']).size().unstack()
burnout_course.plot(kind='bar', figsize=(12, 5))
plt.title('Burnout Level by Course')
plt.xlabel('Course')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Burnout Level')
plt.tight_layout()
plt.show()

# High burnout proportion by course
burnout_prop = df[df['burnout_level']=='High'].groupby('course').size() / df.groupby('course').size()
print('High burnout proportion by course:')
print(burnout_prop.sort_values(ascending=False))

In [ ]:
# Q6: Screen Time Hours Distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['screen_time_hours'].hist(bins=30, color='coral', edgecolor='white')
plt.title('Screen Time Hours Distribution')
plt.xlabel('Screen Time Hours')

plt.subplot(1, 2, 2)
plt.scatter(df['screen_time_hours'], df['depression_score'], alpha=0.1, s=5)
plt.title('Screen Time vs Depression Score')
plt.xlabel('Screen Time Hours')
plt.ylabel('Depression Score')
plt.tight_layout()
plt.show()

In [ ]:
# Q7: Violin plot - CGPA by Internet Quality
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x='internet_quality', y='cgpa',
               order=['Poor', 'Average', 'Good'], palette='muted')
plt.title('CGPA Distribution by Internet Quality')
plt.xlabel('Internet Quality')
plt.ylabel('CGPA')
plt.tight_layout()
plt.show()

In [ ]:
# Q8: Heatmap - Correlation Matrix
plt.figure(figsize=(14, 10))
num_cols = ['age', 'daily_study_hours', 'daily_sleep_hours', 'screen_time_hours',
            'anxiety_score', 'depression_score', 'academic_pressure_score',
            'financial_stress_score', 'social_support_score',
            'physical_activity_hours', 'attendance_percentage', 'cgpa']
corr_matrix = df[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap - All Numerical Variables')
plt.tight_layout()
plt.show()

print('Features most correlated with cgpa:')
print(corr_matrix['cgpa'].sort_values(ascending=False))
print()
print('Features most correlated with anxiety_score:')
print(corr_matrix['anxiety_score'].sort_values(ascending=False))

In [ ]:
# Q9: Pairplot
sample_df = df[['daily_study_hours', 'screen_time_hours', 'cgpa', 'anxiety_score',
                'sleep_quality']].sample(1000, random_state=42)
sns.pairplot(sample_df, hue='sleep_quality', plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot - Key Variables', y=1.02)
plt.show()

In [ ]:
# Q10: Boxplot - Academic Pressure by Burnout Level
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='burnout_level', y='academic_pressure_score',
            order=['Low', 'Medium', 'High'], palette='Reds')
plt.title('Academic Pressure Score by Burnout Level')
plt.xlabel('Burnout Level')
plt.ylabel('Academic Pressure Score')
plt.tight_layout()
plt.show()

print('Mean academic pressure by burnout level:')
print(df.groupby('burnout_level')['academic_pressure_score'].mean())

---
# Section 4: Model Building — Binary Classification

In [ ]:
# Target Variable banao
df_model = df.copy()

# high_burnout: 1 = High, 0 = Low/Medium
df_model['high_burnout'] = (df_model['burnout_level'] == 'High').astype(int)

print('Target variable high_burnout:')
print(df_model['high_burnout'].value_counts())
print()
print('Kyun yeh binarization chose ki:')
print('Hum specifically predict karna chahte hain ki koun HIGH risk mein hai')
print('Binary classification simple aur actionable hai')

In [ ]:
# Feature Selection
print('=== Feature Selection ===')

# Encoding karo pehle
df_model['sleep_quality_enc'] = df_model['sleep_quality'].map({'Poor': 0, 'Average': 1, 'Good': 2})
df_model['stress_level_enc'] = df_model['stress_level'].map({'Low': 0, 'Medium': 1, 'High': 2})
df_model['internet_quality_enc'] = df_model['internet_quality'].map({'Poor': 0, 'Average': 1, 'Good': 2})
df_model['gender_enc'] = df_model['gender'].map({'Male': 0, 'Female': 1, 'Other': 2})
df_model = pd.get_dummies(df_model, columns=['course', 'year'], drop_first=True)

# Features (X)
drop_cols = ['student_id', 'burnout_level', 'high_burnout',
             'gender', 'sleep_quality', 'stress_level', 'internet_quality']
X = df_model.drop(columns=drop_cols)
y = df_model['high_burnout']

print(f'b. Target (y): high_burnout')
print(f'c. Dropped columns: {drop_cols}')
print('   student_id drop kiya - sirf ID hai')
print('   burnout_level drop kiya - data leakage hoga (target se direct related)')
print(f'\na. Features (X) shape: {X.shape}')
print('Features:', X.columns.tolist()[:10], '...')

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('=== Train-Test Split ===')
print(f'Training set  : {X_train.shape[0]:,} rows')
print(f'Testing set   : {X_test.shape[0]:,} rows')
print()
print('a. random_state=42 kyun zaroori hai:')
print('   Taki har baar same split mile -- results reproducible rahen')
print()
print('b. Splitting ka purpose:')
print('   Model training data pe seekhta hai')
print('   Test data pe check karte hain ki unseen data pe kitna acha kaam karta hai')

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('\nFeature scaling done!')

In [ ]:
# Model Training
print('=== Logistic Regression Model ===')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

train_acc = lr_model.score(X_train_scaled, y_train)
print(f'a. max_iter=1000 kyun: Model ko converge hone ke liye zyada iterations chahiye')
print(f'   Default 100 iterations pe warning aa sakti thi')
print()
print(f'b. Training Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)')

---
# Section 5: Model Evaluation

In [ ]:
# Predictions & Metrics
y_pred = lr_model.predict(X_test_scaled)
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print('=== a. Model Performance Metrics ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred):.4f}')
print()
print('Metrics ka matlab:')
print('  Accuracy  = Total sahi predictions / Total predictions')
print('  Precision = Jinhe High predict kiya unme se kitne sach mein High the')
print('  Recall    = Jinhe actually High tha unme se kitne pakde')
print('  F1-Score  = Precision aur Recall ka balance')

In [ ]:
# b. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not High (0)', 'High (1)'],
            yticklabels=['Not High (0)', 'High (1)'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print('Confusion Matrix samjhao:')
print(f'True Negative  (TN): {cm[0][0]} -- Sahi predict kiya Not High')
print(f'False Positive (FP): {cm[0][1]} -- Galat predict kiya High (actually Not High tha)')
print(f'False Negative (FN): {cm[1][0]} -- Galat predict kiya Not High (actually High tha)')
print(f'True Positive  (TP): {cm[1][1]} -- Sahi predict kiya High')

In [ ]:
# ROC-AUC Analysis
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', label=f'ROC Curve (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

print(f'a. AUC Score: {auc_score:.4f}')
print('   AUC = 1.0 matlab perfect model')
print('   AUC = 0.5 matlab random guessing jitna')
print(f'   Hamara AUC {auc_score:.2f} -- model theek kaam kar raha hai')
print()
print('b. Agar AUC < 0.7 hota toh:')
print('   - Features aur target mein koi strong relation nahi')
print('   - Zyada feature engineering ki zaroorat hai')
print('   - Dataset mein noise zyada hai')

In [ ]:
# Classification Report
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Not High (0)', 'High (1)']))
print()
print('Analysis:')
print('Class 0 (Not High burnout) usually better predict hota hai')
print('Kyunki iska training data zyada hota hai (Low + Medium combined)')

---
# Section 6: Model Interpretation & Critical Thinking

In [ ]:
# Coefficient Analysis
print('=== a. Logistic Regression Coefficients ===')
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print('Top 5 features (HIGH burnout badhate hain):')
print(coef_df.head())
print()
print('Top 5 features (HIGH burnout ghatate hain):')
print(coef_df.tail())

# Plot
plt.figure(figsize=(10, 6))
top_features = pd.concat([coef_df.head(8), coef_df.tail(8)])
colors = ['red' if c > 0 else 'steelblue' for c in top_features['Coefficient']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.title('Logistic Regression Coefficients')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

print()
print('b. academic_pressure_score ka effect (non-technical explanation):')
print('   Jitna zyada academic pressure, utna zyada burnout ka khatra.')
print('   Positive coefficient = pressure badhne se burnout probability barhti hai.')

In [ ]:
# Class Imbalance & SMOTE
print('=== Class Imbalance Check ===')
print(y.value_counts())
print(f'Imbalance ratio: {y.value_counts()[0]/y.value_counts()[1]:.2f}')
print()

# SMOTE apply karo
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)
print(f'After SMOTE - Training set size: {len(y_train_sm):,}')
print(f'Class distribution after SMOTE: {pd.Series(y_train_sm).value_counts().to_dict()}')

# SMOTE ke baad model train karo
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_sm, y_train_sm)
y_pred_smote = lr_smote.predict(X_test_scaled)

print()
print('=== Original Model vs SMOTE Model ===')
print(f'Original Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'SMOTE Accuracy    : {accuracy_score(y_test, y_pred_smote):.4f}')
print(f'Original F1       : {f1_score(y_test, y_pred):.4f}')
print(f'SMOTE F1          : {f1_score(y_test, y_pred_smote):.4f}')

In [ ]:
# Preprocessing Impact - Without Scaling
print('=== Without Feature Scaling ===')
lr_no_scale = LogisticRegression(max_iter=1000, random_state=42)
lr_no_scale.fit(X_train, y_train)
y_pred_ns = lr_no_scale.predict(X_test)
print(f'Accuracy without scaling: {accuracy_score(y_test, y_pred_ns):.4f}')
print(f'Accuracy with scaling   : {accuracy_score(y_test, y_pred):.4f}')
print()
print('Scaling matter karta hai kyunki:')
print('  - Bina scaling ke, zyada range wale features dominant ho jaate hain')
print('  - Gradient descent slow ya diverge ho sakta hai')
print('  - Coefficients misleading ho jaate hain')

In [ ]:
# Model Improvement Strategies
print('=== Model Improvement Strategies ===')
print()
print('a. 3 strategies:')
print('   1. Feature Engineering: Naye features banao jaise stress_to_support_ratio')
print('   2. Hyperparameter Tuning: GridSearchCV se C parameter tune karo')
print('   3. Ensemble Methods: Random Forest ya XGBoost try karo')

print()
print('b. L1 (Lasso) Regularization:')
lr_l1 = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=42)
lr_l1.fit(X_train_scaled, y_train)
y_pred_l1 = lr_l1.predict(X_test_scaled)

zero_coef = (lr_l1.coef_[0] == 0).sum()
print(f'   Features with zero coefficient (L1): {zero_coef}')
print(f'   L1 Accuracy: {accuracy_score(y_test, y_pred_l1):.4f}')
print(f'   Original Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print('   L1 automatically unimportant features zero kar deta hai')
print('   Yeh feature selection ka kaam karta hai')

In [ ]:
# Cross-Validation
print('=== 5-Fold Cross Validation ===')
cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'CV Scores     : {cv_scores}')
print(f'Mean Accuracy : {cv_scores.mean():.4f}')
print(f'Std Deviation : {cv_scores.std():.4f}')
print()
print('Std Deviation ka matlab:')
print('  Low std (< 0.02) = model stable hai, consistent performance')
print('  High std (> 0.05) = model unstable hai, different data pe alag results')

In [ ]:
# Random Forest Comparison
print('=== Random Forest vs Logistic Regression ===')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print(f'                  Logistic Reg   Random Forest')
print(f'Accuracy        :   {accuracy_score(y_test, y_pred):.4f}          {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision       :   {precision_score(y_test, y_pred):.4f}          {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall          :   {recall_score(y_test, y_pred):.4f}          {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1-Score        :   {f1_score(y_test, y_pred):.4f}          {f1_score(y_test, y_pred_rf):.4f}')
print()
print('Random Forest usually better hota hai kyunki:')
print('  - Multiple decision trees ka ensemble hai')
print('  - Non-linear relationships bhi pakad sakta hai')
print('  - Overfitting kam hota hai')

---
## Summary & Key Findings

Is assignment mein humne Student Mental Health & Academic Performance dataset par kaam kiya:

**Key Findings:**
- Dataset mein 150,000 students hain aur 20 features hain
- Koi missing values ya duplicates nahi the — dataset already clean tha
- Study hours aur CGPA ka correlation bahut weak hai (-0.005)
- Academic pressure aur stress burnout ke strongest predictors hain
- Logistic Regression ne achha performance diya
- Random Forest ne thoda better results diye

**Model Limitations:**
- Logistic Regression sirf linear relationships pakad sakta hai
- Is dataset mein burnout equally distributed hai — real life mein aisa nahi hota
- Aur features jaise family background, mental health history add karne se model improve ho sakta hai